In [1]:
# =============================================================
# FASE 2 - PIPELINE ETL | Consumo energético en planta siderúrgica
# Celda 1: Importación de librerías y configuración general
# =============================================================

import pandas as pd          # Manipulación de tablas (DataFrames)
import numpy as np           # Cálculos numéricos
from pathlib import Path     # Manejo de rutas que funciona en Windows, Mac y Linux

# --- Reproducibilidad (exigida por la política académica del curso) ---
# Fijamos una semilla aleatoria para que cualquier proceso con azar
# (muestreos, modelos de la Fase 4) dé siempre el mismo resultado.
SEED = 42
np.random.seed(SEED)

# --- Rutas del proyecto ---
# El notebook vive en notebooks/, por eso subimos un nivel ("..")
# para llegar a la carpeta data/.
BASE_DIR      = Path("..")
RAW_PATH      = BASE_DIR / "data" / "raw" / "Steel_industry.csv"
PROCESSED_DIR = BASE_DIR / "data" / "processed"

# Mostrar todas las columnas al imprimir tablas (por defecto pandas oculta algunas)
pd.set_option("display.max_columns", None)

print("Versión de pandas:", pd.__version__)
print("¿Existe el archivo?", RAW_PATH.exists())

Versión de pandas: 3.1.1
¿Existe el archivo? True


In [2]:
# =============================================================
# Celda 2: Ingesta (Extract)
# =============================================================

# encoding="utf-8-sig" elimina el carácter invisible (BOM) que trae
# el archivo al inicio. Sin esto, la primera columna se llamaría
# "\ufeffDate_Time" y df["Date_Time"] daría error.
df_raw = pd.read_csv(RAW_PATH, encoding="utf-8-sig")

# Guardamos df_raw INTACTO como foto del estado original.
# Trabajamos sobre una copia (df). Así podremos comparar "antes vs. después"
# para el diccionario de datos y el reporte de calidad.
df = df_raw.copy()

print("Dimensiones (filas, columnas):", df.shape)
print("Nombre de la primera columna:", repr(df.columns[0]))
df.head()

Dimensiones (filas, columnas): (35041, 11)
Nombre de la primera columna: 'Date_Time'


,Date_Time,Usage_kWh,Lagging_Current_Reactive.Power_kVarh,Leading_Current_Reactive_Power_kVarh,CO2(tCO2),Lagging_Current_Power_Factor,Leading_Current_Power_Factor,NSM,WeekStatus,Day_Of_Week,Load_Type
0,01-01-2018 00:15,3.17,2.95,0.0,0.0,73.21,100.0,900,Weekday,Monday,Light_Load
1,01-01-2018 00:30,4.00,4.46,0.0,0.0,66.77,100.0,1800,Weekday,Monday,Light_Load
2,01-01-2018 00:45,3.24,3.28,0.0,0.0,70.28,100.0,2700,Weekday,Monday,Light_Load
3,01-01-2018 01:00,3.31,3.56,0.0,0.0,68.09,100.0,3600,Weekday,Monday,Light_Load
4,01-01-2018 01:15,3.82,4.50,0.0,0.0,64.72,100.0,4500,Weekday,Monday,Light_Load


In [3]:
# =============================================================
# Celda 3: Diagnóstico inicial de tipos de datos
# =============================================================

# info() muestra el tipo de cada columna, cuántos valores no nulos tiene
# y cuánta memoria ocupa la tabla.
df_raw.info(memory_usage="deep")

# Guardamos esta foto de tipos para el diccionario "antes" (lo usaremos al final)
tipos_antes = df_raw.dtypes.astype(str)

<class 'pandas.DataFrame'>
RangeIndex: 35041 entries, 0 to 35040
Data columns (total 11 columns):
 #   Column                                Non-Null Count  Dtype  
---  ------                                --------------  -----  
 0   Date_Time                             35041 non-null  str    
 1   Usage_kWh                             35041 non-null  float64
 2   Lagging_Current_Reactive.Power_kVarh  35041 non-null  float64
 3   Leading_Current_Reactive_Power_kVarh  35041 non-null  float64
 4   CO2(tCO2)                             35041 non-null  float64
 5   Lagging_Current_Power_Factor          35041 non-null  float64
 6   Leading_Current_Power_Factor          35041 non-null  float64
 7   NSM                                   35041 non-null  int64  
 8   WeekStatus                            35041 non-null  str    
 9   Day_Of_Week                           35041 non-null  str    
 10  Load_Type                             35041 non-null  str    
dtypes: float64(6), int64(1), s

In [4]:
# =============================================================
# Celda 4: Conversión de Date_Time a formato datetime
# =============================================================

# Indicamos el formato EXACTO: día-mes-año hora:minuto.
# ¿Por qué no dejar que pandas lo adivine? Porque "01-02-2018" puede leerse
# como 1 de febrero o como 2 de enero. Con el formato explícito eliminamos
# esa ambigüedad.
# errors="coerce": si alguna fecha no se puede leer, queda como NaT
# (fecha nula) en vez de detener el programa, y así podemos contarlas.
df["Date_Time"] = pd.to_datetime(
    df["Date_Time"],
    format="%d-%m-%Y %H:%M",
    errors="coerce"
)

print("Fechas que no se pudieron convertir:", df["Date_Time"].isna().sum())
print("Primera fecha:", df["Date_Time"].min())
print("Última fecha :", df["Date_Time"].max())

Fechas que no se pudieron convertir: 0
Primera fecha: 2018-01-01 00:00:00
Última fecha : 2018-12-31 23:45:00


In [5]:
# =============================================================
# Celda 5: Corrección de las marcas de medianoche (00:00)
# =============================================================

# EVIDENCIA DEL PROBLEMA: miramos el cambio de día entre el 1 y el 2 de enero.
print("Evidencia ANTES de corregir:")
print(df.loc[93:97, ["Date_Time", "NSM", "Day_Of_Week"]])

# Interpretación:
# - El primer registro del archivo es 01-01-2018 00:15, no 00:00.
#   Eso indica que cada marca de tiempo es el FINAL de un intervalo de 15 min
#   (00:15 = consumo entre 00:00 y 00:15).
# - Por tanto, el intervalo 23:45-24:00 del 1 de enero debería marcarse como
#   02-01-2018 00:00. Pero el archivo lo marca como 01-01-2018 00:00,
#   que es el inicio del mismo día: la fecha va un día atrasada.

# CORRECCIÓN: a los registros de las 00:00 les sumamos un día.
es_medianoche = (df["Date_Time"].dt.hour == 0) & (df["Date_Time"].dt.minute == 0)
print("\nRegistros de medianoche a corregir:", es_medianoche.sum())

df.loc[es_medianoche, "Date_Time"] += pd.Timedelta(days=1)

# VERIFICACIÓN: la diferencia entre registros consecutivos debería ser
# siempre de 15 minutos.
print("\nDiferencias entre registros consecutivos DESPUÉS de corregir:")
print(df["Date_Time"].diff().value_counts())

Evidencia ANTES de corregir:
             Date_Time    NSM Day_Of_Week
93 2018-01-01 23:30:00  84600      Monday
94 2018-01-01 23:45:00  85500      Monday
95 2018-01-01 00:00:00      0      Monday
96 2018-01-02 00:15:00    900     Tuesday
97 2018-01-02 00:30:00   1800     Tuesday

Registros de medianoche a corregir: 365

Diferencias entre registros consecutivos DESPUÉS de corregir:
Date_Time
0 days 00:15:00      35039
-1 days +20:00:00        1
Name: count, dtype: int64


In [6]:
# =============================================================
# Celda 6: Conversión de variables de texto a tipo categórico
# =============================================================

# Load_Type tiene un ORDEN natural: liviana < media < máxima.
# Declararlo como categoría ordenada hace que las gráficas y tablas
# respeten ese orden en vez de usar el orden alfabético.
orden_carga = pd.CategoricalDtype(
    categories=["Light_Load", "Medium_Load", "Maximum_Load"],
    ordered=True
)
df["Load_Type"] = df["Load_Type"].astype(orden_carga)

# Los días también tienen orden (lunes a domingo, no alfabético).
orden_dias = pd.CategoricalDtype(
    categories=["Monday", "Tuesday", "Wednesday", "Thursday",
                "Friday", "Saturday", "Sunday"],
    ordered=True
)
df["Day_Of_Week"] = df["Day_Of_Week"].astype(orden_dias)

# WeekStatus solo tiene 2 valores y no necesita orden.
df["WeekStatus"] = df["WeekStatus"].astype("category")

# Verificamos que la conversión no creó nulos
# (aparecerían si algún valor no coincide con las categorías declaradas).
print("Nulos tras convertir categorías:",
      df[["Load_Type", "Day_Of_Week", "WeekStatus"]].isna().sum().sum())

df.info(memory_usage="deep")

Nulos tras convertir categorías: 0
<class 'pandas.DataFrame'>
RangeIndex: 35041 entries, 0 to 35040
Data columns (total 11 columns):
 #   Column                                Non-Null Count  Dtype         
---  ------                                --------------  -----         
 0   Date_Time                             35041 non-null  datetime64[us]
 1   Usage_kWh                             35041 non-null  float64       
 2   Lagging_Current_Reactive.Power_kVarh  35041 non-null  float64       
 3   Leading_Current_Reactive_Power_kVarh  35041 non-null  float64       
 4   CO2(tCO2)                             35041 non-null  float64       
 5   Lagging_Current_Power_Factor          35041 non-null  float64       
 6   Leading_Current_Power_Factor          35041 non-null  float64       
 7   NSM                                   35041 non-null  int64         
 8   WeekStatus                            35041 non-null  category      
 9   Day_Of_Week                           35041 non-

In [7]:
# =============================================================
# Celda 7: Bitácora de calidad + diagnóstico de valores nulos
# =============================================================

# Lista donde anotaremos cada chequeo: qué encontramos, qué hicimos
# y cuántas filas se afectaron. Al final será el REPORTE DE CALIDAD
# que pide el entregable de la Fase 2.
registro_calidad = []

# Contamos nulos por columna y su porcentaje sobre el total de filas
nulos = df.isna().sum()
tabla_nulos = nulos.to_frame("nulos")
tabla_nulos["porcentaje"] = (nulos / len(df) * 100).round(2)
print(tabla_nulos)

registro_calidad.append({
    "Chequeo": "Valores nulos",
    "Hallazgo": f"{int(nulos.sum())} nulos en {df.shape[1]} columnas",
    "Acción": "Ninguna (no requiere imputación)",
    "Filas afectadas": int(nulos.sum())
})

                                      nulos  porcentaje
Date_Time                                 0         0.0
Usage_kWh                                 0         0.0
Lagging_Current_Reactive.Power_kVarh      0         0.0
Leading_Current_Reactive_Power_kVarh      0         0.0
CO2(tCO2)                                 0         0.0
Lagging_Current_Power_Factor              0         0.0
Leading_Current_Power_Factor              0         0.0
NSM                                       0         0.0
WeekStatus                                0         0.0
Day_Of_Week                               0         0.0
Load_Type                                 0         0.0


In [8]:
# =============================================================
# Celda 8: Duplicados y completitud de la serie de tiempo
# =============================================================

# 1) Duplicados COMPLETOS: filas idénticas en todas las columnas
dup_completos = df.duplicated().sum()
print("Filas idénticas en todo:", dup_completos)

# 2) Duplicados por MARCA DE TIEMPO: en una serie de tiempo, cada
#    instante debe aparecer UNA sola vez, aunque los valores difieran.
dup_fecha = df["Date_Time"].duplicated(keep=False)   # keep=False marca ambas copias
print("Filas con marca de tiempo repetida:", dup_fecha.sum())
print(df[dup_fecha].T)   # .T voltea la tabla para comparar las dos filas lado a lado

Filas idénticas en todo: 0
Filas con marca de tiempo repetida: 2
                                                    35023                35040
Date_Time                             2018-12-31 20:00:00  2018-12-31 20:00:00
Usage_kWh                                            4.14                 4.15
Lagging_Current_Reactive.Power_kVarh                  0.0                  0.0
Leading_Current_Reactive_Power_kVarh                20.09                 20.1
CO2(tCO2)                                             0.0                  0.0
Lagging_Current_Power_Factor                        100.0                100.0
Leading_Current_Power_Factor                        20.18                20.19
NSM                                                 72000                72000
WeekStatus                                        Weekday              Weekday
Day_Of_Week                                        Monday               Monday
Load_Type                                      Light_Load         

In [9]:
# DECISIÓN: conservamos la primera aparición (la que está en su lugar
# dentro de la secuencia) y eliminamos la copia que quedó al final del archivo.
filas_antes = len(df)
df = (df.drop_duplicates(subset="Date_Time", keep="first")
        .sort_values("Date_Time")     # ordenamos cronológicamente
        .reset_index(drop=True))      # renumeramos las filas 0, 1, 2...
filas_eliminadas = filas_antes - len(df)
print("Filas eliminadas:", filas_eliminadas, "| Filas actuales:", len(df))

# COMPLETITUD: generamos TODOS los intervalos de 15 min que deberían existir
# entre la primera y la última fecha, y buscamos cuáles faltan.
esperadas = pd.date_range(start=df["Date_Time"].min(),
                          end=df["Date_Time"].max(), freq="15min")
faltantes = esperadas.difference(df["Date_Time"])
print("Intervalos esperados:", len(esperadas), "| Faltantes:", len(faltantes))
print("¿Orden cronológico correcto?", df["Date_Time"].is_monotonic_increasing)

registro_calidad.append({
    "Chequeo": "Duplicados",
    "Hallazgo": "1 marca de tiempo repetida (31-12-2018 20:00) con valores casi idénticos",
    "Acción": "Se conservó el registro en secuencia y se eliminó la copia del final",
    "Filas afectadas": int(filas_eliminadas)
})
registro_calidad.append({
    "Chequeo": "Completitud de la serie",
    "Hallazgo": f"{len(df)} de {len(esperadas)} intervalos de 15 min",
    "Acción": "Ninguna (serie completa)",
    "Filas afectadas": len(faltantes)
})

Filas eliminadas: 1 | Filas actuales: 35040
Intervalos esperados: 35040 | Faltantes: 0
¿Orden cronológico correcto? True


In [10]:
# =============================================================
# Celda 9: Detección de fallas de medición (registro en ceros)
# =============================================================

cols_energia = ["Usage_kWh", "Lagging_Current_Reactive.Power_kVarh",
                "Leading_Current_Reactive_Power_kVarh", "CO2(tCO2)"]
cols_fp      = ["Lagging_Current_Power_Factor", "Leading_Current_Power_Factor"]
cols_medicion = cols_energia + cols_fp

# Buscamos filas donde TODAS las mediciones eléctricas valen cero.
# Que el consumo sea cero puede pasar (planta apagada), pero que el factor
# de potencia también sea 0 % es físicamente imposible: sin corriente,
# el factor de potencia no está definido. Es una falla del medidor.
falla = (df[cols_medicion] == 0).all(axis=1)
print("Registros con todas las mediciones en cero:", falla.sum())

# Contexto: vemos los registros vecinos
i = df.index[falla][0]
print(df.loc[i-2:i+2, ["Date_Time", "Usage_kWh", "Lagging_Current_Power_Factor", "Load_Type"]])

Registros con todas las mediciones en cero: 1
                Date_Time  Usage_kWh  Lagging_Current_Power_Factor   Load_Type
29853 2018-11-07 23:30:00       3.60                         50.27  Light_Load
29854 2018-11-07 23:45:00       3.56                         51.66  Light_Load
29855 2018-11-08 00:00:00       0.00                          0.00  Light_Load
29856 2018-11-08 00:15:00       3.64                         51.12  Light_Load
29857 2018-11-08 00:30:00       3.64                         49.61  Light_Load


In [11]:
# DECISIÓN: imputación por interpolación lineal.
# ¿Por qué no eliminar la fila? Porque dejaría un hueco en la serie de
# tiempo, y la serie completa es clave para los perfiles horarios del EDA.
# ¿Por qué interpolar? La carga era liviana y estable antes y después,
# así que el promedio de los vecinos es la estimación más razonable.

# Marcamos la fila ANTES de modificarla (trazabilidad: se puede auditar)
df["Registro_Imputado"] = falla.astype(int)

df.loc[falla, cols_medicion] = np.nan                       # 1) el cero falso pasa a "dato faltante"
df[cols_medicion] = df[cols_medicion].interpolate(method="linear")  # 2) se rellena con los vecinos
df["CO2(tCO2)"] = df["CO2(tCO2)"].round(2)                   # 3) mantenemos la resolución original del CO2

print(df.loc[i-1:i+1, ["Date_Time", "Usage_kWh", "Lagging_Current_Power_Factor"]])
print("Nulos tras imputar:", df.isna().sum().sum())

registro_calidad.append({
    "Chequeo": "Falla de medición",
    "Hallazgo": "1 registro (2018-11-08 00:00) con todas las mediciones en cero, incluido FP = 0 %",
    "Acción": "Interpolación lineal con registros vecinos; marcado en Registro_Imputado",
    "Filas afectadas": int(falla.sum())
})

                Date_Time  Usage_kWh  Lagging_Current_Power_Factor
29854 2018-11-07 23:45:00       3.56                         51.66
29855 2018-11-08 00:00:00       3.60                         51.39
29856 2018-11-08 00:15:00       3.64                         51.12
Nulos tras imputar: 0


In [12]:
# =============================================================
# Celda 10: Validación con reglas de negocio
# =============================================================
# Un dato puede no ser nulo ni duplicado y aun así ser imposible.
# Aquí verificamos que los datos respeten la física y la lógica del proceso.

# NSM esperado = hora×3600 + minuto×60
nsm_calc = df["Date_Time"].dt.hour * 3600 + df["Date_Time"].dt.minute * 60

# Fin de semana esperado según el día
finde_calc = np.where(df["Day_Of_Week"].isin(["Saturday", "Sunday"]), "Weekend", "Weekday")

# Factor de potencia teórico: FP = P / √(P² + Q²) × 100
# (P = energía activa en kWh, Q = energía reactiva inductiva en kVarh)
fp_calc = 100 * df["Usage_kWh"] / np.sqrt(df["Usage_kWh"]**2 +
                                          df["Lagging_Current_Reactive.Power_kVarh"]**2)

reglas = {
    "Energía o CO2 negativos": (df[cols_energia] < 0).any(axis=1).sum(),
    "Factor de potencia fuera de 0-100 %": ((df[cols_fp] < 0) | (df[cols_fp] > 100)).any(axis=1).sum(),
    "NSM no coincide con la hora": (nsm_calc != df["NSM"]).sum(),
    "WeekStatus no coincide con el día": (finde_calc != df["WeekStatus"]).sum(),
    "FP en atraso no coincide con la fórmula (> 0,1 pts)":
        ((fp_calc - df["Lagging_Current_Power_Factor"]).abs() > 0.1).sum(),
}
tabla_reglas = pd.Series(reglas, name="Registros que violan la regla").astype(int)
print(tabla_reglas)

registro_calidad.append({
    "Chequeo": "Reglas de consistencia física y lógica",
    "Hallazgo": f"{int(tabla_reglas.sum())} violaciones en {len(reglas)} reglas",
    "Acción": "Ninguna (datos coherentes)",
    "Filas afectadas": int(tabla_reglas.sum())
})

Energía o CO2 negativos                                0
Factor de potencia fuera de 0-100 %                    0
NSM no coincide con la hora                            0
WeekStatus no coincide con el día                      0
FP en atraso no coincide con la fórmula (> 0,1 pts)    0
Name: Registros que violan la regla, dtype: int64


In [13]:
# =============================================================
# Celda 11: Detección de outliers (método IQR)
# =============================================================

def detectar_outliers_iqr(serie):
    """Devuelve una máscara True/False de outliers y los límites del IQR."""
    q1 = serie.quantile(0.25)
    q3 = serie.quantile(0.75)
    iqr = q3 - q1
    lim_inf = q1 - 1.5 * iqr
    lim_sup = q3 + 1.5 * iqr
    mascara = (serie < lim_inf) | (serie > lim_sup)
    return mascara, lim_inf, lim_sup

# Aplicamos la función a todas las variables de medición
resumen = []
for col in cols_medicion:
    mascara, lim_inf, lim_sup = detectar_outliers_iqr(df[col])
    resumen.append({
        "Variable": col,
        "Límite inferior": round(lim_inf, 2),
        "Límite superior": round(lim_sup, 2),
        "Outliers": int(mascara.sum()),
        "% del total": round(mascara.mean() * 100, 1)
    })
tabla_outliers = pd.DataFrame(resumen)
tabla_outliers

,Variable,Límite inferior,Límite superior,Outliers,% del total
0,Usage_kWh,-68.86,123.29,328,0.9
1,Lagging_Current_Reactive.Power_kVarh,-28.21,53.15,1059,3.0
2,Leading_Current_Reactive_Power_kVarh,-3.14,5.22,7759,22.1
3,CO2(tCO2),-0.03,0.05,437,1.2
4,Lagging_Current_Power_Factor,9.77,152.58,0,0.0
5,Leading_Current_Power_Factor,99.25,100.45,8326,23.8


In [14]:
# ¿En qué tipo de carga ocurren los outliers de consumo?
mask_uso, _, lim_sup_uso = detectar_outliers_iqr(df["Usage_kWh"])
print(f"Umbral de pico: {lim_sup_uso:.1f} kWh por intervalo de 15 min\n")
print(df.loc[mask_uso, "Load_Type"].value_counts())

# Proporción de las variables capacitivas que vale exactamente cero o 100
print("\n% de intervalos sin reactiva capacitiva:",
      round((df["Leading_Current_Reactive_Power_kVarh"] == 0).mean() * 100, 1))
print("% de intervalos con CO2 = 0:", round((df["CO2(tCO2)"] == 0).mean() * 100, 1))

Umbral de pico: 123.3 kWh por intervalo de 15 min

Load_Type
Maximum_Load    149
Medium_Load     146
Light_Load       33
Name: count, dtype: int64

% de intervalos sin reactiva capacitiva: 67.4
% de intervalos con CO2 = 0: 59.9


In [15]:
# DECISIÓN: no eliminamos ni recortamos outliers; los MARCAMOS.
# La columna Pico_Consumo (1 = pico) servirá en el EDA y en el modelado.
df["Pico_Consumo"] = mask_uso.astype(int)
print(df["Pico_Consumo"].value_counts())

registro_calidad.append({
    "Chequeo": "Outliers (IQR) en Usage_kWh",
    "Hallazgo": f"{int(mask_uso.sum())} registros > {lim_sup_uso:.1f} kWh (0,9 %), 90 % en carga media o máxima",
    "Acción": "Se conservan (picos reales de producción) y se marcan en Pico_Consumo",
    "Filas afectadas": int(mask_uso.sum())
})
registro_calidad.append({
    "Chequeo": "Outliers en reactiva capacitiva, FP en adelanto y CO2",
    "Hallazgo": "Distribuciones concentradas en 0 o 100 y CO2 redondeado a 0,01 t",
    "Acción": "Se conservan; el IQR no es adecuado para estas distribuciones",
    "Filas afectadas": 0
})

Pico_Consumo
0    34712
1      328
Name: count, dtype: int64


In [16]:
# =============================================================
# Celda 12: Reporte de calidad de datos (entregable Fase 2)
# =============================================================
reporte_calidad = pd.DataFrame(registro_calidad)
print(f"Filas: {len(df_raw)} (original) → {len(df)} (limpio)")
reporte_calidad

Filas: 35041 (original) → 35040 (limpio)


,Chequeo,Hallazgo,Acción,Filas afectadas
0,Valores nulos,0 nulos en 11 columnas,Ninguna (no requiere imputación),0
1,Duplicados,1 marca de tiempo repetida (31-12-2018 20:00) ...,Se conservó el registro en secuencia y se elim...,1
2,Completitud de la serie,35040 de 35040 intervalos de 15 min,Ninguna (serie completa),0
3,Falla de medición,1 registro (2018-11-08 00:00) con todas las me...,Interpolación lineal con registros vecinos; ma...,1
4,Reglas de consistencia física y lógica,0 violaciones en 5 reglas,Ninguna (datos coherentes),0
5,Outliers (IQR) en Usage_kWh,"328 registros > 123.3 kWh (0,9 %), 90 % en car...",Se conservan (picos reales de producción) y se...,328
6,"Outliers en reactiva capacitiva, FP en adelant...",Distribuciones concentradas en 0 o 100 y CO2 r...,Se conservan; el IQR no es adecuado para estas...,0


In [18]:
# =============================================================
# FASE 2 - BLOQUE 3: INGENIERÍA DE CARACTERÍSTICAS
# Celda 13: Variables de tiempo
# =============================================================

# Recordatorio del Bloque 1: Date_Time marca el FINAL de cada intervalo.
# El registro de las 00:15 es el consumo entre 00:00 y 00:15.
# Para asignar bien la hora y el día, usamos el INICIO del intervalo.
# Así, el registro de las 00:00 (consumo de 23:45 a 24:00) queda en la
# hora 23 del día anterior, que es donde realmente ocurrió.
inicio_intervalo = df["Date_Time"] - pd.Timedelta(minutes=15)

df["Fecha"] = inicio_intervalo.dt.normalize()   # fecha sin hora (día operativo)
df["Hora"]  = inicio_intervalo.dt.hour          # 0 a 23
df["Mes"]   = inicio_intervalo.dt.month         # 1 a 12

# Turnos típicos de una planta que opera 24 horas en 3 turnos de 8 horas:
#   Mañana: 06:00-13:59 | Tarde: 14:00-21:59 | Noche: 22:00-05:59
condiciones = [
    df["Hora"].between(6, 13),
    df["Hora"].between(14, 21),
]
df["Turno"] = np.select(condiciones, ["Mañana", "Tarde"], default="Noche")
df["Turno"] = df["Turno"].astype(
    pd.CategoricalDtype(["Mañana", "Tarde", "Noche"], ordered=True)
)

# Verificación: con el inicio del intervalo, el día calculado debe coincidir
# SIEMPRE con la columna Day_Of_Week original.
dia_calculado = inicio_intervalo.dt.day_name()
print("Días que no coinciden con Day_Of_Week:",
      (dia_calculado != df["Day_Of_Week"].astype(str)).sum())
print(df["Turno"].value_counts())

Días que no coinciden con Day_Of_Week: 0
Turno
Mañana    11680
Tarde     11680
Noche     11680
Name: count, dtype: int64


In [19]:
# =============================================================
# Celda 14: KPIs de demanda y calidad eléctrica
# =============================================================

# --- Demanda promedio en kW ---
# Usage_kWh es ENERGÍA consumida en 15 minutos. La DEMANDA (potencia)
# es energía / tiempo: kWh / 0,25 h = kWh × 4.
# Las empresas de energía suelen cobrar también por la demanda máxima,
# por eso este KPI importa para la factura.
df["Demanda_kW"] = df["Usage_kWh"] * 4

# --- Ratio reactiva / activa (Q/P) ---
# Indica cuánta energía "improductiva" circula por cada kWh útil.
# Ratio 0 = ideal; ratio alto = mucha carga inductiva (motores, transformadores).
# np.where evita dividir entre cero si algún intervalo no tuviera consumo.
df["Ratio_Reactiva_Activa"] = np.where(
    df["Usage_kWh"] > 0,
    df["Lagging_Current_Reactive.Power_kVarh"] / df["Usage_kWh"],
    0
)

# --- Reactiva penalizable ---
# SUPUESTO (basado en la regulación colombiana, CREG 015 de 2018):
# se cobra la energía reactiva inductiva que SUPERA el 50 % de la activa.
# Ese umbral equivale aproximadamente a un factor de potencia de 0,9.
# Simplificación: aplicamos la regla a cada intervalo de 15 min.
UMBRAL_REACTIVA = 0.5

df["Reactiva_Penalizable_kVarh"] = (
    df["Lagging_Current_Reactive.Power_kVarh"] - UMBRAL_REACTIVA * df["Usage_kWh"]
).clip(lower=0)   # clip: si el resultado es negativo (no hay exceso), queda en 0

df["Penalizado"] = (df["Reactiva_Penalizable_kVarh"] > 0).astype(int)

print(f"% de intervalos con penalización: {df['Penalizado'].mean()*100:.1f} %\n")
print(df.groupby("Load_Type", observed=True)[
    ["Demanda_kW", "Ratio_Reactiva_Activa", "Penalizado"]
].mean().round(2))

% de intervalos con penalización: 53.3 %

              Demanda_kW  Ratio_Reactiva_Activa  Penalizado
Load_Type                                                  
Light_Load         34.51                   1.06        0.79
Medium_Load       153.78                   0.31        0.20
Maximum_Load      237.06                   0.42        0.34


In [20]:
# =============================================================
# Celda 15: KPIs de costo
# =============================================================

# SUPUESTO ILUSTRATIVO: tarifa plana de 0,10 USD/kWh.
# El dataset no incluye precios, así que la tarifa es un PARÁMETRO:
# si conoces la tarifa real, cambia solo este número y todo se recalcula.
# Simplificación: la reactiva penalizable se cobra al mismo precio del kWh.
TARIFA_USD_KWH = 0.10

df["Costo_Energia_USD"]      = df["Usage_kWh"] * TARIFA_USD_KWH
df["Costo_Penalizacion_USD"] = df["Reactiva_Penalizable_kVarh"] * TARIFA_USD_KWH
df["Costo_Total_USD"]        = df["Costo_Energia_USD"] + df["Costo_Penalizacion_USD"]

costos = df[["Costo_Energia_USD", "Costo_Penalizacion_USD", "Costo_Total_USD"]].sum()
print(costos.round(0))
print(f"\nLa penalización equivale al "
      f"{costos['Costo_Penalizacion_USD'] / costos['Costo_Energia_USD'] * 100:.1f} % "
      f"del costo de energía")

Costo_Energia_USD          95964.0
Costo_Penalizacion_USD      8092.0
Costo_Total_USD           104056.0
dtype: float64

La penalización equivale al 8.4 % del costo de energía


In [21]:
# =============================================================
# Celda 16: KPI de emisiones
# =============================================================

# Problema del Bloque 2: el CO2 viene redondeado a 0,01 t, por eso el 60 %
# de los intervalos marca 0. A nivel de intervalo es poco útil.
# Solución: calculamos el FACTOR DE EMISIÓN de la planta con los totales
# anuales (donde el redondeo ya no pesa) y lo aplicamos a cada intervalo.
factor_emision = df["CO2(tCO2)"].sum() / df["Usage_kWh"].sum() * 1000   # tCO2 por MWh
print(f"Factor de emisión: {factor_emision:.3f} tCO2/MWh")

# kWh × (tCO2/MWh) = kg de CO2   (porque 1 t/MWh = 1 kg/kWh)
df["CO2_Estimado_kg"] = df["Usage_kWh"] * factor_emision

print(f"CO2 original total : {df['CO2(tCO2)'].sum():.2f} t")
print(f"CO2 estimado total : {df['CO2_Estimado_kg'].sum()/1000:.2f} t")

Factor de emisión: 0.421 tCO2/MWh
CO2 original total : 403.81 t
CO2 estimado total : 403.81 t


In [22]:
# =============================================================
# Celda 17: Factor de carga diario
# =============================================================

# Factor de carga = demanda promedio del día / demanda máxima del día.
#   Cercano a 1 → consumo parejo, la capacidad contratada se aprovecha bien.
#   Cercano a 0 → consumo con picos esporádicos, se paga capacidad que casi no se usa.
resumen_diario = df.groupby("Fecha")["Demanda_kW"].agg(["mean", "max"])
factor_carga = resumen_diario["mean"] / resumen_diario["max"]

# .map() pega el valor diario a cada uno de los 96 registros de ese día
df["Factor_Carga_Diario"] = df["Fecha"].map(factor_carga)

print(factor_carga.describe().round(2))
print("\nDimensiones finales:", df.shape)
print("Nulos totales:", df.isna().sum().sum())

count    365.00
mean       0.38
std        0.18
min        0.08
25%        0.27
50%        0.33
75%        0.39
max        0.86
dtype: float64

Dimensiones finales: (35040, 26)
Nulos totales: 0
